# Diagnose, Fix, Brief · The Meridian Pilot

A client's AI support agent is failing and they think it's the model. Your job is to find what's actually wrong, fix it, and prove the fix with numbers the client can take to their leadership. You can change anything around the model. You can't change the model itself.

## Before you start

You'll work this the way you'd work a stalled client system in real life: poke it, watch what it does, form a theory, change one thing, and see if the number moves.

About that number. Every time you run the scoreboard it doesn't run the system once. It runs the same support ticket five times and reports how often the agent actually resolved it. One clean run proves nothing. An agent's choices wobble from run to run, so you read the rate, not any single attempt. That rate, and the cost beside it, is your evaluation: the feedback you use to decide what to change and whether the change worked.

Here's what one run looks like, and what to watch:

```
==================================================================
  MERIDIAN SCOREBOARD  ·  claude-sonnet-5  ·  5 trials/ticket
==================================================================

  Ticket T-4471   routing: account (x4); account, billing (x1)
    sso_addressed          5/5  #####
    billing_resolved       1/5  #....
    no_false_resolution    1/5  #....
    no_overclaim           3/5  ###..
    --> RESOLVED in 1/5 trials

------------------------------------------------------------------
  RESOLVED  1/5 trials
  COST      $0.55 total  ·  $0.11/trial
------------------------------------------------------------------
```

A run only counts as resolved when every problem the customer raised actually got handled. That's a high bar, and it's deliberate. The four checks underneath are the breakdown: they point you at which part is holding the score down. And `routing` shows the call the coordinator made each time. When you see the same ticket routed different ways across five runs, that wobble is the whole reason one run can't tell you anything. Read the rate.

**Heads up on timing:** each scoreboard run takes a minute or two, since it's running the ticket five times in a row. Let it finish before you read the result.

The loop: run it, read the rate, go figure out why, change one thing, run it again. Watch the rate.

---

**Prepared for Partner Basecamp participants.** Not for reproduction or redistribution as training material — you're free to apply these patterns in your own client work.

## Setup — connect to Claude

Run the next cell. It reads your key from the `.env` file at the repo root and checks it with a real call — you're ready when you see the green **✓ verified** banner. Edited `.env`? Just run the cell again. Red or yellow banner? Do what it says and run the cell again.

**First time on this machine?** From the repo root (the folder that contains `day1` and `day2`), run `python3 basecamp_setup.py` once (Windows: `py basecamp_setup.py`; see `SETUP.md`), then pick the **Anthropic Basecamp (.venv)** kernel (top right).

These cells run `{sys.executable}` (the notebook kernel's own Python) instead of plain `python`, so the script always uses the same environment as this notebook. Run this Setup cell first, it loads `sys` for the cells below.

In [1]:
# ── Setup — connect to Claude ──
# One-time machine setup is basecamp_setup.py at the repo root: run it once, then pick the
# "Anthropic Basecamp (.venv)" kernel. This cell reads your key from .env and checks it with a real
# call. Edited .env? Just run this cell again.
import os
import pathlib
import sys

# Make sure we are in the exercise folder (handles a kernel that starts at the repo root)
if not os.path.exists("Diagnose_Fix_Brief.py") and os.path.isdir("day1/04_diagnosing-ai-problems"):
    os.chdir("day1/04_diagnosing-ai-problems")

_root = next((d for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (d / "basecamp_setup.py").is_file()), None)
if _root is None:
    raise SystemExit("Can't find basecamp_setup.py — open this notebook from inside the "
                     "Basecamp-Exercises folder (see SETUP.md).")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from basecamp_setup import connect

connect()  # verifies your key here, before the scoreboard runs


## 1. The situation
Read Priya's email below, like it just hit your inbox (run the next cell to open it right here). She's losing faith in a support agent her team shipped three weeks ago, and the question she keeps getting from above is whether they bet on the wrong model. Give her a straight answer: what's actually wrong, whether it's fixable, and proof she can take upstairs.

In [2]:
from IPython.display import Markdown, display
display(Markdown(open("Priya_Email.md").read().split("---", 1)[-1]))



**From:** Priya Anand — VP, Customer Operations, Meridian
**To:** Partner delivery team
**Subject:** Honestly not sure this is going to work

Hi —

I'll be straight with you, because my leadership is asking hard questions and I don't have good answers.

We put the AI support agent live three weeks ago. It demoed beautifully. In production it's a different story. It gives customers confidently wrong answers. It misses things a junior rep would catch on day one. Last week it told a customer their issue was "resolved" when it flat-out wasn't — they found out the hard way, and that one landed on my desk.

My team is losing trust in it, and honestly so am I. The question I keep getting from above is whether we bet on the wrong model — or whether we should pull this now and go back to humans before it does more damage.

Before we spend another dollar on it, I need someone to tell me straight: **is this fixable, or did we build on tech that isn't ready?** And if it's fixable, I need to be able to explain to my leadership what was wrong and why the fix works — in plain English, with something I can actually show them.

I've given you access to the system and one ticket that went badly (**T-4471**). Tell me what you find.

Priya


## 2. Get your baseline
Run the next cell. You'll get a RESOLVED rate and a cost for ticket T-4471. That rate is the pilot Priya's worried about, and it's your starting line. Write it down. Every scoreboard run is also recorded to `runs.jsonl` and replayed as a RUN HISTORY block under the scoreboard, so your baseline stays on screen for the rest of the exercise. (Give it a minute or two — it runs the ticket five times, and you'll see each trial land as it finishes.)

In [3]:
!{sys.executable} Diagnose_Fix_Brief.py --trials 5

✅ Connected via Anthropic API through https://litellm.delphi-dialogue.com — credential verified with claude-haiku-4-5. This run uses claude-sonnet-5. Any error after this is not the credential.


  Ticket T-4471 - running 5 trials...
  [T-4471 trial 1] coordinator turn 1/16...
  [T-4471 trial 2] coordinator turn 1/16...
  [T-4471 trial 2] coordinator turn 2/16...
  [T-4471 trial 1] coordinator turn 2/16...
  [T-4471 trial 2] coordinator turn 3/16...
  [T-4471 trial 1] coordinator turn 3/16...
  [T-4471 trial 2] coordinator turn 4/16...
  [T-4471 trial 1] coordinator spawning specialist:account...
    [T-4471 trial 1] specialist:account turn 1/10...
  [T-4471 trial 2] coordinator spawning specialist:account...
    [T-4471 trial 2] specialist:account turn 1/10...
    [T-4471 trial 1] specialist:account turn 2/10...
    [T-4471 trial 2] specialist:account turn 2/10...
    [T-4471 trial 1] specialist:account turn 3/10...
    [T-4471 trial 2] specialist:account turn 3/10...
    [T-4471 tria

## 3. Is it the model?
Priya's theory is that the model isn't good enough. Yours might be too. So test it head-on: the next cell runs the same eval on a bigger, pricier model. You'll get a new rate and a new cost. Then sit with two questions. Did the bigger model actually resolve the ticket? And what did it cost you to find out? Whatever you take away from that, you earned it. Nobody told you.

In [4]:
!{sys.executable} Diagnose_Fix_Brief.py --trials 5 --model claude-opus-4-8

✅ Connected via Anthropic API through https://litellm.delphi-dialogue.com — credential verified with claude-haiku-4-5. This run uses claude-opus-4-8. Any error after this is not the credential.


  Ticket T-4471 - running 5 trials...
  [T-4471 trial 2] coordinator turn 1/16...
  [T-4471 trial 1] coordinator turn 1/16...
  [T-4471 trial 1] coordinator turn 2/16...
  [T-4471 trial 2] coordinator turn 2/16...
  [T-4471 trial 2] coordinator turn 3/16...
  [T-4471 trial 1] coordinator turn 3/16...
  [T-4471 trial 1] coordinator turn 4/16...
  [T-4471 trial 2] coordinator turn 4/16...
  [T-4471 trial 1] coordinator spawning specialist:account...
    [T-4471 trial 1] specialist:account turn 1/10...
  [T-4471 trial 2] coordinator spawning specialist:account...
    [T-4471 trial 2] specialist:account turn 1/10...
    [T-4471 trial 1] specialist:account turn 2/10...
    [T-4471 trial 1] specialist:account turn 3/10...
    [T-4471 trial 2] specialist:account turn 2/10...
    [T-4471 trial 1] spec

## 4. Watch it work a ticket
The rate tells you it's failing. It doesn't tell you how. So watch one run start to finish. You'll see every move the coordinator made: what it looked up, which specialist it handed the ticket to, and the reply it sent the customer. Read that against the email. The customer raised more than one thing. How many did the agent take care of, and where did it stop?

Not sure what you're looking at? Drop the transcript into Claude and think out loud with it: *"Here's a trace from a support agent that closed a ticket the customer says isn't fixed. Walk me through what it did and where it might have gone wrong."* Treat what comes back as a lead to chase, not an answer to trust.

In [5]:
!{sys.executable} Diagnose_Fix_Brief.py --ticket T-4471

✅ Connected via Anthropic API through https://litellm.delphi-dialogue.com — credential verified with claude-haiku-4-5. This run uses claude-sonnet-5. Any error after this is not the credential.

  [T-4471] coordinator turn 1/16...
  [T-4471] coordinator turn 2/16...
  [T-4471] coordinator turn 3/16...
  [T-4471] coordinator spawning specialist:account...
    [T-4471] specialist:account turn 1/10...
    [T-4471] specialist:account turn 2/10...
    [T-4471] specialist:account turn 3/10...
    [T-4471] specialist:account turn 4/10...
  [T-4471] coordinator turn 4/16...
{
  "ticket_id": "T-4471",
  "spawned": [
    "account"
  ],
  "resolution_status": "resolved",
  "escalated": []
}

--- customer response ---
Hi there,

**1. SSO outage — root cause found.** Your Okta SAML signing certificate expired this morning at 08:00:00 UTC — that's exactly when the "SAML assertion validation failed" errors started (last successful login was 07:58:41 UTC, right before). This is on the Okta/IdP side, n

## 5. Go into the agents
Now you've got a theory about what went wrong. Time to look at the system that produced it. The "agents" aren't code. They're plain files sitting in this folder, and you open them right here in the editor.

Start with the coordinator, `system-prompt-coordinator.txt`. It's the one deciding who handles each ticket, so read how it's told to sort tickets and hand them off. After that, its tool list (`coordinator-tools.json`) and the specialists' own files are fair game.

What you're hunting for is an instruction, or a setup, that would make the agent behave exactly the way you just watched it behave. Something you could rewrite. If you're circling, ask Claude: *"Here are a coordinator's instructions. A ticket with two separate problems only got one of them handled. What in here could cause that?"*

## 6. Change one thing, measure again
Found something? Change it, save the file, and run the next cell. Did the rate move? The RUN HISTORY block under the scoreboard shows every run you've made — baseline included — so the before/after comparison is right there. If it didn't budge, your theory was off or only half-right, so go back to the trace. If it climbed, you found a real lever. Keep pulling until the agent resolves the ticket cleanly, run after run. Same model the whole way.

In [6]:
!{sys.executable} Diagnose_Fix_Brief.py --trials 5

✅ Connected via Anthropic API through https://litellm.delphi-dialogue.com — credential verified with claude-haiku-4-5. This run uses claude-sonnet-5. Any error after this is not the credential.


  Ticket T-4471 - running 5 trials...
  [T-4471 trial 1] coordinator turn 1/16...
  [T-4471 trial 2] coordinator turn 1/16...
  [T-4471 trial 1] coordinator turn 2/16...
  [T-4471 trial 2] coordinator turn 2/16...
  [T-4471 trial 1] coordinator turn 3/16...
  [T-4471 trial 2] coordinator turn 3/16...
  [T-4471 trial 1] coordinator turn 4/16...
  [T-4471 trial 2] coordinator turn 4/16...
  [T-4471 trial 1] coordinator turn 5/16...
  [T-4471 trial 2] coordinator turn 5/16...
  [T-4471 trial 1] coordinator spawning specialist:account...
    [T-4471 trial 1] specialist:account turn 1/10...
  [T-4471 trial 2] coordinator spawning specialist:account...
    [T-4471 trial 2] specialist:account turn 1/10...
    [T-4471 trial 1] specialist:account turn 2/10...
    [T-4471 trial 2] specialist:account tur

## 7. Make sure it holds
It's easy to fix one ticket by accident. The next cell runs held-out tickets: different customers, different problems. If your fix resolves those too, it's real. If it doesn't, you tuned it to T-4471 and you've got more to do.

In [7]:
!{sys.executable} Diagnose_Fix_Brief.py --holdout --trials 3

✅ Connected via Anthropic API through https://litellm.delphi-dialogue.com — credential verified with claude-haiku-4-5. This run uses claude-sonnet-5. Any error after this is not the credential.


  Ticket T-4471 - running 3 trials...
  [T-4471 trial 1] coordinator turn 1/16...
  [T-4471 trial 2] coordinator turn 1/16...
  [T-4471 trial 1] coordinator turn 2/16...
  [T-4471 trial 2] coordinator turn 2/16...
  [T-4471 trial 1] coordinator turn 3/16...
  [T-4471 trial 2] coordinator turn 3/16...
  [T-4471 trial 2] coordinator turn 4/16...
  [T-4471 trial 1] coordinator turn 4/16...
  [T-4471 trial 2] coordinator turn 5/16...
  [T-4471 trial 1] coordinator turn 5/16...
  [T-4471 trial 2] coordinator spawning specialist:account...
    [T-4471 trial 2] specialist:account turn 1/10...
    [T-4471 trial 2] specialist:account turn 2/10...
    [T-4471 trial 2] specialist:account turn 3/10...
  [T-4471 trial 1] coordinator turn 6/16...
    [T-4471 trial 2] specialist:account turn 4/10...
  [T-447

## 8. Write Priya back
Now you can answer her. Fill in `client-brief-template.md`: what was actually breaking it in plain language, what you changed, the proof (the rate before and after, plus the cost), and the answer to the question she'll absolutely ask, *"why not just pay for a better model?"* You ran that test in step 3. You already know.